# State-space models from scratch: recurrence, convolution, scan
**LLMs From Basics · AR.3 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/ar-3-ssm-scan.ipynb)

In the lesson an SSM carried a fixed-size state through a sequence. In NumPy you'll build:

1. A discretized SSM, run as a recurrence
2. The same SSM as a convolution, and a check that both agree
3. A parallel scan that computes every state in about $\log_2 L$ rounds
4. A selective (Mamba-style) SSM whose step size depends on the input, so it can *choose* what to remember
5. A memory comparison: KV cache versus SSM state

Each exercise has a `# TODO` stub followed by a check cell; solutions are at the end. CPU only, well under a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Discretize, then run the recurrence

We use a **diagonal** $A$ (stored as a vector of $N$ negative numbers), as S4D and Mamba do. Zero-order-hold discretization with step $\Delta$ gives, element by element,

$$\bar A = e^{\Delta A}, \qquad \bar B = \frac{e^{\Delta A} - 1}{A}\,B$$

and the recurrence for one input channel $x_1, \dots, x_L$ (starting from $h_0 = 0$) is

$$h_t = \bar A \odot h_{t-1} + \bar B\,x_t, \qquad y_t = C \cdot h_t$$

Write `discretize(A, B, delta)` and `ssm_recurrent(Abar, Bbar, C, x)`.

In [ ]:
def discretize(A, B, delta):
    # TODO: return Abar, Bbar (both shape (N,))
    raise NotImplementedError

def ssm_recurrent(Abar, Bbar, C, x):
    h = np.zeros_like(Abar, dtype=float)
    y = np.zeros(len(x))
    # TODO: loop over t, update h, read out y[t]
    return y

In [ ]:
Ab, Bb = discretize(np.array([-1.0]), np.array([1.0]), np.log(2))
assert np.allclose(Ab, [0.5]) and np.allclose(Bb, [0.5])
# The lesson's worked example: Abar = 0.5, Bbar = 1, C = 2, x = (1, 0, 0, 1)
y = ssm_recurrent(np.array([0.5]), np.array([1.0]), np.array([2.0]), np.array([1.0, 0, 0, 1]))
assert np.allclose(y, [2, 1, 0.5, 2.25]), y
# The faded example: Abar = 0.9, Bbar = 0.5, C = 1, x = (2, 0, 1) -> y3 = 1.31
assert np.isclose(ssm_recurrent(np.array([0.9]), np.array([0.5]), np.array([1.0]), np.array([2.0, 0, 1]))[-1], 1.31)
print("Exercise 1 passed:", y)

## 2. The convolution view (S4)

Unrolling the recurrence gives $y_t = \sum_{j=0}^{t-1} \bar K_j\,x_{t-j}$ with the kernel $\bar K_j = C \cdot (\bar A^{\,j} \odot \bar B)$. Write `ssm_kernel(Abar, Bbar, C, L)` and `ssm_conv(K, x)` (a causal convolution: `np.convolve` returns too many outputs, so keep the first `len(x)`).

In [ ]:
def ssm_kernel(Abar, Bbar, C, L):
    # TODO: K[j] = sum over the N state entries of C * Abar**j * Bbar, for j = 0..L-1
    raise NotImplementedError

def ssm_conv(K, x):
    # TODO
    raise NotImplementedError

In [ ]:
assert np.allclose(ssm_kernel(np.array([0.5]), np.array([1.0]), np.array([2.0]), 4), [2, 1, 0.5, 0.25])
N, L = 8, 300
A = -np.linspace(0.2, 3.0, N); B = rng.normal(size=N); Cv = rng.normal(size=N)
Abar, Bbar = discretize(A, B, 0.1)
x = rng.normal(size=L)
y_rec = ssm_recurrent(Abar, Bbar, Cv, x)
y_conv = ssm_conv(ssm_kernel(Abar, Bbar, Cv, L), x)
assert np.allclose(y_rec, y_conv), "recurrence and convolution must agree"
print("Exercise 2 passed: max difference", np.abs(y_rec - y_conv).max())

## 3. A parallel scan

Mamba can't use a fixed kernel, because its parameters change every step. It uses the fact that steps of the form $h_t = a_t h_{t-1} + b_t$ compose:

$$(a_1, b_1) \text{ then } (a_2, b_2) \;=\; (a_2 a_1,\; a_2 b_1 + b_2)$$

That operation is associative, so we can combine neighbours in rounds: in round $r$, every position $t \ge 2^r$ absorbs the combined pair from position $t - 2^r$ (the Hillis–Steele scan). After $\lceil \log_2 L \rceil$ rounds, position $t$ holds the composition of steps $1..t$, and its $b$ is $h_t$.

Write `parallel_scan(a, b)` for scalar sequences, using NumPy array operations for each round (no loop over $t$). Return `h` and the number of rounds.

In [ ]:
def parallel_scan(a, b):
    a, b = a.astype(float).copy(), b.astype(float).copy()
    L, shift, rounds = len(a), 1, 0
    while shift < L:
        # TODO: for positions t >= shift, combine (a[t-shift], b[t-shift]) followed by (a[t], b[t]).
        #       Compute both new arrays from the OLD values before overwriting.
        shift *= 2
        rounds += 1
    return b, rounds

In [ ]:
def sequential_scan(a, b):
    h, out = 0.0, np.zeros(len(a))
    for t in range(len(a)):
        h = a[t] * h + b[t]
        out[t] = h
    return out

a_t = rng.uniform(0.5, 1.0, size=1000); b_t = rng.normal(size=1000)
h_par, rounds = parallel_scan(a_t, b_t)
assert np.allclose(h_par, sequential_scan(a_t, b_t))
assert rounds == 10, rounds                                   # ceil(log2 1000) = 10
print(f"Exercise 3 passed: 1000 steps in {rounds} parallel rounds")

## 4. A selective SSM

Now make the step size depend on the input, as Mamba does: $\Delta_t = \mathrm{softplus}(w\,x_t + c)$, then $\bar A_t = e^{\Delta_t A}$ and $\bar B_t = \frac{\bar A_t - 1}{A} B$ for every step. With a large positive $w$ and a negative $c$, ordinary small inputs get a tiny $\Delta_t$ ("skip me") while a large input gets a big one ("reset and store me").

Write `selective_ssm(x, A, B, C, w, c)`. Use `np.logaddexp(0, z)` for a stable softplus.

In [ ]:
def selective_ssm(x, A, B, C, w, c):
    h = np.zeros_like(A, dtype=float)
    y = np.zeros(len(x))
    for t in range(len(x)):
        # TODO: delta_t from x[t], then Abar_t, Bbar_t, then update h and read y[t]
        pass
    return y

In [ ]:
N = 4
A4 = -np.array([0.5, 1.0, 1.5, 2.0]); B4 = np.ones(N); C4 = np.ones(N) / N
x_sig = np.zeros(120); x_sig[10] = 5.0                        # one important token, then 109 filler tokens
y_sel = selective_ssm(x_sig, A4, B4, C4, w=20.0, c=-8.0)       # filler gets delta = softplus(-8) ≈ 0.0003
softplus_inv_1 = np.log(np.e - 1)                              # c giving delta = 1 for every token
y_fix = selective_ssm(x_sig, A4, B4, C4, w=0.0, c=softplus_inv_1)
keep_sel = y_sel[-1] / y_sel[10]
keep_fix = y_fix[-1] / y_fix[10]
print(f"memory of the important token after 109 filler tokens: selective {keep_sel:.3f}, fixed step {keep_fix:.2e}")
assert keep_sel > 0.9, "the selective SSM should keep most of the important token"
assert keep_fix < 1e-6, "a fixed step size of 1 should forget it"
print("Exercise 4 passed")

## 5. Memory: KV cache versus SSM state

Write `kv_cache_bytes(layers, kv_heads, head_dim, bytes_per, tokens)` (keys **and** values, every layer, every token) and `ssm_state_bytes(layers, d_inner, N, bytes_per, d_conv=4)` (an $N$-sized state per channel plus Mamba's convolution buffer of `d_conv` recent inputs per channel; no dependence on length).

In [ ]:
def kv_cache_bytes(layers, kv_heads, head_dim, bytes_per, tokens):
    # TODO
    raise NotImplementedError

def ssm_state_bytes(layers, d_inner, N, bytes_per, d_conv=4):
    # TODO
    raise NotImplementedError

In [ ]:
kv = kv_cache_bytes(32, 8, 128, 2, 1_000_000)                  # Llama 3.1 8B, 1M tokens, bf16
st = ssm_state_bytes(64, 5120, 16, 4)                           # Mamba-2.8B, fp32
assert kv == 131_072_000_000 and st == 26_214_400
print(f"KV cache at 1M tokens: {kv / 1e9:.0f} GB · SSM state: {st / 1e6:.1f} MB · ratio {kv / st:,.0f}x")
print("Exercise 5 passed")

## Experiment

1. **Memory versus length.** The plot below shows both memories on log axes. Where do they cross? (Early on the state is bigger.)
2. **Kernels and $\Delta$.** Plot `ssm_kernel` for $\Delta$ = 0.01, 0.1 and 1. Which one remembers longest, and which reacts most to each input?
3. **Recurrence versus convolution speed.** Time both on $L$ = 20,000 with an FFT convolution (`np.fft.rfft`). The recurrence is a Python loop here; on a GPU, the scan is what makes it fast.

In [ ]:
tokens = np.logspace(1, 6, 50)
plt.figure(figsize=(6, 3.5))
plt.loglog(tokens, kv_cache_bytes(32, 8, 128, 2, tokens), label="KV cache (Llama 3.1 8B, bf16)")
plt.loglog(tokens, np.full_like(tokens, ssm_state_bytes(64, 5120, 16, 4)), label="SSM state (Mamba-2.8B, fp32)")
plt.xlabel("tokens in the sequence"); plt.ylabel("bytes per sequence"); plt.legend(); plt.grid(alpha=0.3, which="both")
plt.show()

plt.figure(figsize=(6, 3.5))
for d in (0.01, 0.1, 1.0):
    Ab, Bb = discretize(A, B, d)
    plt.plot(ssm_kernel(Ab, Bb, Cv, 200), label=f"Δ = {d}")
plt.xlabel("lag j"); plt.ylabel("kernel K_j"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

L_big = 20_000
x_big = rng.normal(size=L_big)
Ab, Bb = discretize(A, B, 0.1)
t0 = time.perf_counter(); y1 = ssm_recurrent(Ab, Bb, Cv, x_big); t1 = time.perf_counter()
K = ssm_kernel(Ab, Bb, Cv, L_big)
n_fft = 2 * L_big
y2 = np.fft.irfft(np.fft.rfft(x_big, n_fft) * np.fft.rfft(K, n_fft), n_fft)[:L_big]
t2 = time.perf_counter()
print(f"recurrence {t1 - t0:.3f}s · FFT convolution {t2 - t1:.4f}s · max difference {np.abs(y1 - y2).max():.2e}")

## Solutions

Try the exercises first. Each solution defines the same function as its stub, so running a solution cell and then the check cell again should pass.

### Solution 1

In [ ]:
def discretize(A, B, delta):
    Abar = np.exp(delta * A)
    Bbar = (Abar - 1.0) / A * B
    return Abar, Bbar

def ssm_recurrent(Abar, Bbar, C, x):
    h = np.zeros_like(Abar, dtype=float)
    y = np.zeros(len(x))
    for t in range(len(x)):
        h = Abar * h + Bbar * x[t]
        y[t] = C @ h
    return y

### Solution 2

In [ ]:
def ssm_kernel(Abar, Bbar, C, L):
    powers = Abar[None, :] ** np.arange(L)[:, None]          # (L, N)
    return powers @ (C * Bbar)

def ssm_conv(K, x):
    return np.convolve(x, K)[: len(x)]

### Solution 3

In [ ]:
def parallel_scan(a, b):
    a, b = a.astype(float).copy(), b.astype(float).copy()
    L, shift, rounds = len(a), 1, 0
    while shift < L:
        a_prev, b_prev = a[:-shift].copy(), b[:-shift].copy()
        b[shift:] = a[shift:] * b_prev + b[shift:]
        a[shift:] = a[shift:] * a_prev
        shift *= 2
        rounds += 1
    return b, rounds

### Solution 4

In [ ]:
def selective_ssm(x, A, B, C, w, c):
    h = np.zeros_like(A, dtype=float)
    y = np.zeros(len(x))
    for t in range(len(x)):
        delta = np.logaddexp(0.0, w * x[t] + c)
        Abar = np.exp(delta * A)
        Bbar = (Abar - 1.0) / A * B
        h = Abar * h + Bbar * x[t]
        y[t] = C @ h
    return y

### Solution 5

In [ ]:
def kv_cache_bytes(layers, kv_heads, head_dim, bytes_per, tokens):
    return 2 * layers * kv_heads * head_dim * bytes_per * tokens

def ssm_state_bytes(layers, d_inner, N, bytes_per, d_conv=4):
    return layers * d_inner * (N + d_conv) * bytes_per